# TPWRS PRP Publications by Year: Six Affinity Bands

In [ ]:
from pathlib import Path

import pandas as pd
import plotly.express as px

repo_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
benchmark_root = repo_root / "data" / "results" / "affinity_benchmark"
mapping_path = repo_root / "data" / "processed" / "ra_grouping_ra2025.csv"
band_order = ["Low-", "Low+", "Medium-", "Medium+", "High-", "High+"]
programme_order = ["IBR", "Stability", "DER", "CROF", "System Services", "Planning"]


def load_programme_mapping():
    programme_mapping = pd.read_csv(mapping_path)[["RA2025", "Grouping"]]
    return (
        programme_mapping
        .dropna()
        .drop_duplicates()
        .rename(columns={"RA2025": "RA2025_ID", "Grouping": "PRP_Name"})
    )


def assign_programme_bands(ra_adjudication_df):
    programme_mapping = load_programme_mapping()
    question_scores = ra_adjudication_df.copy()
    question_scores["RA2025_ID"] = pd.to_numeric(question_scores["RA2025_ID"], errors="coerce")
    question_scores["Final_Affinity"] = pd.to_numeric(question_scores["Final_Affinity"], errors="coerce")
    programme_mapping["RA2025_ID"] = pd.to_numeric(programme_mapping["RA2025_ID"], errors="coerce")
    question_scores = question_scores.merge(programme_mapping, on="RA2025_ID", how="left")
    question_scores = question_scores.dropna(
        subset=["Abstract_Index", "PRP_Name", "RA2025_ID", "Final_Affinity"]
    )
    programme_scores = (
        question_scores
        .sort_values(
            ["Abstract_Index", "PRP_Name", "Final_Affinity", "RA2025_ID"],
            ascending=[True, True, False, True],
        )
        .drop_duplicates(["Abstract_Index", "PRP_Name"], keep="first")
        .rename(
            columns={
                "RA2025_ID": "Highest_Question_ID",
                "RA_Question": "Highest_Question_Text",
                "Final_Affinity": "Highest_Question_Affinity",
            }
        )
    )
    programme_scores["Affinity_Band"] = pd.cut(
        programme_scores["Highest_Question_Affinity"],
        bins=[0, 20, 40, 60, 80, 100, float('inf')],
        labels=band_order,
        right=False,
    )
    return programme_scores


assignments_df = pd.concat(
    [
        assign_programme_bands(pd.read_csv(path)).assign(
            Year=int(path.parent.name.split("_")[1])
        )
        for path in sorted(benchmark_root.glob("tpwrs_*/adjudicated_ra_affinities.csv"))
    ],
    ignore_index=True,
 )

papers_per_year = assignments_df.groupby("Year")["Abstract_Index"].nunique()
coverage_bands = (
    assignments_df
    .groupby(["Year", "PRP_Name", "Affinity_Band"], observed=True)["Abstract_Index"]
    .nunique()
    .rename("Papers")
    .reindex(
        pd.MultiIndex.from_product(
            [papers_per_year.index, programme_order, band_order],
            names=["Year", "PRP_Name", "Affinity_Band"],
        ),
        fill_value=0,
    )
    .reset_index()
    .merge(papers_per_year.rename("Papers_Total"), on="Year")
 )
coverage_bands["Share"] = 100 * coverage_bands["Papers"] / coverage_bands["Papers_Total"]
coverage_bands["Affinity_Band"] = pd.Categorical(
    coverage_bands["Affinity_Band"], categories=band_order, ordered=True
 )


figure = px.area(
    coverage_bands,
    x="Year",
    y="Share",
    color="Affinity_Band",
    facet_col="PRP_Name",
    facet_col_wrap=2,
    category_orders={
        "PRP_Name": programme_order,
        "Affinity_Band": band_order[::-1],
    },
 )
# figure.update_yaxes(range=[0, 80])
figure.show()

In [ ]:
group_highest_scores = (
    assignments_df
    .groupby(["PRP_Name", "Affinity_Band"], observed=True)["Highest_Question_Affinity"]
    .transform("max")
)

highest_scoring_indices_by_prp_band = (
    assignments_df
    .loc[assignments_df["Highest_Question_Affinity"].eq(group_highest_scores)]
    .groupby(["PRP_Name", "Affinity_Band"], observed=True)
    .agg(
        Highest_Score=("Highest_Question_Affinity", "first"),
        Year_Abstract_Question_IDs=(
            "Highest_Question_ID",
            lambda question_ids: list(
                zip(
                    assignments_df.loc[question_ids.index, "Year"],
                    assignments_df.loc[question_ids.index, "Abstract_Index"],
                    question_ids,
                    strict=True,
                )
            ),
        ),
    )
    .reindex(
        pd.MultiIndex.from_product(
            [programme_order, band_order], names=["PRP_Name", "Affinity_Band"]
        )
    )
)
highest_scoring_indices_by_prp_band

In [ ]:
coverage_bands.groupby(["PRP_Name", "Affinity_Band"])["Papers"].sum()

In [ ]:
px.histogram(
    assignments_df,
    x= "Affinity_Band",
    # color = "Year"
    facet_col = "PRP_Name",
    facet_col_wrap=3,
    category_orders={ "Affinity_Band": band_order[::-1]}
)

In [ ]:
assignments_df

In [ ]:
paper_programme_bands = assignments_df.pivot(
    index=["Year", "Abstract_Index"],
    columns="PRP_Name",
    values="Affinity_Band",
)
target_band_order = band_order

source_target_band_summaries = []
for source_prp in programme_order:
    source_prp_is_high = paper_programme_bands[source_prp].isin(["High-", "High+"])

    for target_prp in programme_order:
        if target_prp == source_prp:
            continue

        for affinity_band in target_band_order:
            target_prp_in_band = paper_programme_bands[target_prp].eq(affinity_band)
            papers_with_high_source_and_target_band = source_prp_is_high & target_prp_in_band
            conditional_probability = papers_with_high_source_and_target_band[source_prp_is_high].mean()
            target_prp_baseline_probability = target_prp_in_band.mean()
            lift = (
                conditional_probability / target_prp_baseline_probability
                if target_prp_baseline_probability > 0
                else float("nan")
            )

            source_target_band_summaries.append(
                {
                    "Source_PRP": source_prp,
                    "Target_PRP": target_prp,
                    "Affinity_Band": affinity_band,
                    "Papers_with_High_Source_PRP": source_prp_is_high.sum(),
                    "Papers_with_High_Source_and_Target_PRP_Band": (
                        papers_with_high_source_and_target_band.sum()
                    ),
                    "Probability_of_Target_PRP_Band_given_High_Source_PRP": (
                        conditional_probability
                    ),
                    "Baseline_Probability_of_Target_PRP_Band": target_prp_baseline_probability,
                    "Lift_vs_Target_PRP_Baseline": lift,
                }
            )

source_target_band_diversification_summary = pd.DataFrame(source_target_band_summaries)
source_target_band_diversification_summary

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

source_prp_heatmaps = {}
for source_prp in programme_order:
    source_prp_heatmaps[source_prp] = (
        source_target_band_diversification_summary
        .query("Source_PRP == @source_prp")
        .pivot(
            index="Target_PRP",
            columns="Affinity_Band",
            values="Probability_of_Target_PRP_Band_given_High_Source_PRP",
        )
        .reindex(
            index=[prp for prp in programme_order if prp != source_prp],
            columns=target_band_order,
        )
    )

heatmap_figure = make_subplots(
    rows=3,
    cols=2,
    subplot_titles=[f"High source: {source_prp}" for source_prp in programme_order],
    vertical_spacing=0.1,
)

for index, source_prp in enumerate(programme_order):
    source_prp_heatmap = source_prp_heatmaps[source_prp]
    heatmap_figure.add_trace(
        go.Heatmap(
            z=source_prp_heatmap,
            x=source_prp_heatmap.columns,
            y=source_prp_heatmap.index,
            coloraxis="coloraxis",
            texttemplate="%{z:.1%}",
            textfont={"color": "#0b0b0b"},
            xgap=2,
            ygap=2,
            hovertemplate=(
                f"High source PRP: {source_prp}<br>"
                "Target PRP: %{y}<br>"
                "Affinity band: %{x}<br>"
                "Probability: %{z:.1%}<extra></extra>"
            ),
        ),
        row=index // 2 + 1,
        col=index % 2 + 1,
    )

heatmap_figure.update_layout(
    title="Target Affinity Bands Given a High-Scoring Source PRP",
    height=950,
    template="plotly_white",
    coloraxis={
        "colorscale": [
            [0, "#fcfcfb"],
            [0.5, "#86b6ef"],
            [0.75, "#2a78d6"],
            [1, "#104281"],
        ],
        "cmin": 0,
        "cmax": source_target_band_diversification_summary[
            "Probability_of_Target_PRP_Band_given_High_Source_PRP"
        ].max(),
        "colorbar": {"title": "Probability", "tickformat": ".0%"},
    },
)
heatmap_figure.update_xaxes(title_text="Target affinity band")
heatmap_figure.update_yaxes(title_text="Target PRP")
heatmap_figure.show()

In [ ]:
lift_heatmap_figure = make_subplots(
    rows=3,
    cols=2,
    subplot_titles=[f"High source: {source_prp}" for source_prp in programme_order],
    vertical_spacing=0.1,
)

for index, source_prp in enumerate(programme_order):
    source_prp_lift_heatmap = (
        source_target_band_diversification_summary
        .query("Source_PRP == @source_prp")
        .pivot(
            index="Target_PRP",
            columns="Affinity_Band",
            values="Lift_vs_Target_PRP_Baseline",
        )
        .reindex(
            index=[prp for prp in programme_order if prp != source_prp],
            columns=target_band_order,
        )
    )

    lift_heatmap_figure.add_trace(
        go.Heatmap(
            z=source_prp_lift_heatmap,
            x=source_prp_lift_heatmap.columns,
            y=source_prp_lift_heatmap.index,
            coloraxis="coloraxis",
            texttemplate="%{z:.2f}x",
            textfont={"color": "#0b0b0b"},
            xgap=2,
            ygap=2,
            hovertemplate=(
                f"High source PRP: {source_prp}<br>"
                "Target PRP: %{y}<br>"
                "Affinity band: %{x}<br>"
                "Lift: %{z:.2f}x<extra></extra>"
            ),
        ),
        row=index // 2 + 1,
        col=index % 2 + 1,
    )

lift_heatmap_figure.update_layout(
    title="Lift of Target Affinity Bands Given a High-Scoring Source PRP",
    height=950,
    template="plotly_white",
    coloraxis={
        "colorscale": [
            [0, "#e34948"],
            [0.5, "#fcfcfb"],
            [1, "#2a78d6"],
        ],
        "cmin": 0,
        "cmid": 1,
        "colorbar": {"title": "Lift vs. baseline", "tickformat": ".1f"},
    },
)
lift_heatmap_figure.update_xaxes(title_text="Target affinity band")
lift_heatmap_figure.update_yaxes(title_text="Target PRP")
lift_heatmap_figure.show()